# Analyze Annotation Progress

Reads the Firebase RTDB export and reports alignment stats per video.

In [3]:
import json
from pathlib import Path

ROOT = Path(".").resolve().parent.parent
EXPORT_PATH = ROOT / "data/firebase/latest.json"

with open(EXPORT_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

videos = data.get("videos", {})
video_captions = data.get("video_captions", {})

# Firebase may store videos as list if keys are sequential integers
if isinstance(videos, list):
    videos = {str(i): v for i, v in enumerate(videos) if v is not None}

print(f"{len(videos)} videos, {len(video_captions)} caption sets")

26 videos, 26 caption sets


## Per-Video Stats

In [4]:
total_aligned = 0
total_captions = 0
total_aligned_dur = 0.0
total_dur = 0.0

rows = []

for yt_id, cap_data in video_captions.items():
    captions = cap_data if isinstance(cap_data, list) else cap_data.get("captions", [])
    
    n_total = len(captions)
    n_aligned = 0
    dur_aligned = 0.0
    dur_total = 0.0
    
    for c in captions:
        if not isinstance(c, dict):
            continue
        d = c.get("end_time", 0) - c.get("start_time", 0)
        dur_total += d
        if c.get("aligned"):
            n_aligned += 1
            dur_aligned += d
    
    # Find assigned_to from videos list
    assigned = ""
    for v in videos.values():
        if isinstance(v, dict) and yt_id in v.get("url", ""):
            assigned = v.get("assigned_to", "")
            break
    
    pct = (n_aligned / n_total * 100) if n_total else 0
    rows.append((yt_id, assigned, n_aligned, n_total, pct, dur_aligned, dur_total))
    
    total_aligned += n_aligned
    total_captions += n_total
    total_aligned_dur += dur_aligned
    total_dur += dur_total

# Sort by aligned duration descending
rows.sort(key=lambda r: r[5], reverse=True)

print(f"{'Video ID':<15} {'Assigned':<15} {'Aligned':>8} {'Total':>6} {'%':>6}  {'Dur (min)':>10} {'Total (min)':>12}")
print("-" * 85)
for yt_id, assigned, n_al, n_tot, pct, dur_al, dur_tot in rows:
    print(f"{yt_id:<15} {assigned:<15} {n_al:>8} {n_tot:>6} {pct:>5.1f}%  {dur_al/60:>10.1f} {dur_tot/60:>12.1f}")
print("-" * 85)
pct_total = (total_aligned / total_captions * 100) if total_captions else 0
print(f"{'TOTAL':<15} {'':<15} {total_aligned:>8} {total_captions:>6} {pct_total:>5.1f}%  {total_aligned_dur/60:>10.1f} {total_dur/60:>12.1f}")
print(f"\nAligned: {total_aligned_dur/60:.1f} min ({total_aligned_dur/3600:.2f} hours)")

Video ID        Assigned         Aligned  Total      %   Dur (min)  Total (min)
-------------------------------------------------------------------------------------
d37lwXaSjs4     Volosnka             502    516  97.3%        41.4         41.8
jj5jiyl2mh0     Iriha2025            422    446  94.6%        37.1         38.0
IOflFDS2biE     Iriha2025            508    527  96.4%        35.7         36.5
0ULOz5HM4pA     Volosnka             236    251  94.0%        30.0         31.3
yPYU48eSeBg     LS04071977           294    301  97.7%        27.6         28.0
9NMtlqDBY_s     Volosnka             355    366  97.0%        27.3         27.9
VVjY5HVY0jg     Volosnka             350    366  95.6%        26.9         27.5
4FUDnWC9UJA     Volosnka             315    318  99.1%        25.8         25.9
cNT6ajjEwVU     Iriha2025            169    187  90.4%        25.7         26.9
6O0ZiSgKJNc     Volosnka             249    268  92.9%        22.3         23.5
SG9xYYOLBNI     Iriha2025         